# PySTAC capabilities with live DaFab

This notebook starts from the DaFab STAC root only, then discovers collections, facet catalogs, item links, and representative items through STAC links exposed by the service.

## Notebook index

1. **Setup**: imports, DaFab endpoint, STAC I/O, table rendering, helpers, schema validator.
2. **Root-first navigation**: open the root, inspect links, discover collections, summarize collections, walk the catalog tree.
3. **Item navigation**: collect item links, fetch representative Items, inspect item-first links, resolve DaFab-local targets, recover reverse facet membership, inspect assets and DaFab metadata.
4. **Validation and serialization**: STAC validation, DaFab profile checks, round-trip serialization.


## What this notebook demonstrates

- Open the live DaFab STAC root with PySTAC.
- Discover collections and facet catalogs through `child` links.
- Discover Items through collection/catalog `item` links.
- Fetch representative Items discovered from the live graph.
- Navigate from fetched Items through declared outbound links such as `root`, `collection`, `related`, and `derived_from`.
- Resolve DaFab-local link targets and reconstruct reverse facet membership from the root-driven tree scan.
- Inspect item assets, provenance links, and `dafab:*` metadata blocks.
- Run STAC core validation, PySTAC full validation, and lightweight DaFab profile checks.
- Show where PySTAC stops and DaFab/Rucio or `pystac-client` functionality would be needed.


## Current PySTAC boundaries for DaFab

| Capability | Why this is outside PySTAC core | What would be needed |
|---|---|---|
| Standard STAC API `/search` | PySTAC represents STAC JSON objects; API Item Search is a client/service interaction. | Use `pystac-client` if DaFab exposes a STAC API Item Search endpoint; use DaFab's enhanced filter API for DaFab/Rucio-specific filters. |
| DaFab enhanced filters | They are DaFab/Rucio-specific query semantics, not STAC core navigation semantics. | A DaFab endpoint/client method, or a translator for the subset that maps cleanly to STAC API query/filter. |
| Reverse item-to-facet lookup from an Item alone * | DaFab facet membership can be many-to-many, while STAC `parent`/`collection` links are not suitable for multiple facet backlinks. | Traverse from the STAC root and match catalog `item` links, expose a DaFab lookup endpoint, or index facet membership for query/search. |
| Rucio DID create/attach/detach, hierarchy repair, replicas | These are Rucio data-management operations, not STAC object-model operations. | Use authenticated DaFab service or Rucio APIs. PySTAC can prepare or inspect the STAC JSON payloads around those operations. |
| Remote writes, asset transfer, or BulkMetaDoc patching | PySTAC's default writer serializes STAC JSON locally; it does not call DaFab/Rucio mutation or transfer endpoints. | DaFab/Rucio client methods, plus a custom `StacIO` only if STAC JSON reads/writes should be routed through that service. |
| Enforce DaFab publication rules | STAC allows many graph shapes that DaFab intentionally rejects. | DaFab profile schemas or custom validators covering source/derived membership, facet placement, and required assets. |
| Typed helpers for `dafab:*` fields | Custom fields are preserved, but PySTAC has no DaFab extension package. | Publish a DaFab STAC extension schema and optionally add PySTAC extension helper classes. |
| Typed Processing extension helpers | Processing extension fields are preserved, but the PySTAC package used here does not provide a typed Processing extension helper. | Add a PySTAC Processing extension helper or continue treating those fields as raw STAC extension properties. |
| Python certificate trust for DaFab | PySTAC delegates HTTPS to Python/urllib3, so reads can fail if the Python certificate store cannot validate the DaFab chain. | Install/update Python certificates, configure the CA bundle, or use the custom `StacIO`; avoid disabling TLS verification except for controlled local diagnosis. |

* DaFab facet catalogs are navigation/index views, not necessarily the single canonical parent of an Item. The same Item may be listed under multiple facets, such as basin, anomaly type, season, or other derived classifications. Encoding those memberships as Item backlinks with `rel="parent"` or `rel="collection"` would eventually make the Item multi-parented or multi-collected, which conflicts with STAC hierarchy semantics. DaFab therefore keeps facet membership in the downward direction, with catalogs/collections linking to Items via `rel="item"`, and reverse membership is derived by scanning/querying that tree.


## 1. Setup

### 1.1 Imports

Load the small set of Python modules used by the notebook and print the PySTAC version being exercised.


In [34]:
from __future__ import annotations

import logging
import warnings
from html import escape
from urllib.parse import urlparse

import pystac
from pystac.serialization import identify_stac_object_type
from pystac.stac_io import DefaultStacIO
from pystac.validation import validate_dict
from pystac.validation.local_validator import LocalValidator

# Keep the rendered notebook focused on graph results rather than library noise.
warnings.filterwarnings("ignore", category=DeprecationWarning)
logging.getLogger("pystac.validation.stac_validator").setLevel(logging.CRITICAL)

# Rich tables are optional so the helpers also work outside Jupyter.
try:
    from IPython.display import HTML, display
except Exception:
    HTML = display = None

# Record the library context alongside the live service results.
print("PySTAC", pystac.__version__)
print("Default STAC version", pystac.get_stac_version())

PySTAC 1.15.0-rc.0
Default STAC version 1.1.0


### 1.2 DaFab endpoint and HTTPS settings

Use the public DaFab STAC root directly. The TLS switch is explicit because some Python certificate stores reject the DaFab chain even when the browser and `curl` work.


In [35]:
# This root document is the only hard-coded navigation entry point.
DAFAB_STAC_URL = "https://dafab.cern.ch/stac"
# Supply a CA bundle when Python does not trust the DaFab certificate chain.
CA_BUNDLE = None
# Disabled only for this diagnostic run. Production access should verify TLS.
VERIFY_TLS = False

### 1.3 STAC I/O

Register the only custom PySTAC I/O hook used here: HTTP reads with the selected TLS behavior. Navigation still uses normal PySTAC objects and links.


In [36]:
# Adapt only HTTP transport while parsing and traversal remain standard PySTAC.
class DaFabStacIO(DefaultStacIO):
    def __init__(self, ca_bundle=None, verify_tls=True):
        super().__init__()
        self.ca_bundle = ca_bundle
        self.verify_tls = verify_tls

    def read_text_from_href(self, href: str) -> str:
        # Preserve DefaultStacIO behavior for local files and fixtures.
        if not href.startswith(("http://", "https://")):
            return super().read_text_from_href(href)

        import urllib3

        if not self.verify_tls:
            urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
        # Apply the same certificate policy to every document reached by a link.
        http = urllib3.PoolManager(
            cert_reqs="CERT_REQUIRED" if self.verify_tls else "CERT_NONE",
            ca_certs=self.ca_bundle,
        )
        with http.request(
            "GET",
            href,
            headers={"User-Agent": f"pystac/{pystac.__version__}"},
            preload_content=False,
        ) as response:
            if response.status >= 400:
                raise RuntimeError(f"GET {href} returned HTTP {response.status}")
            return response.read().decode("utf-8")


# Use one reader for the root and a factory for links resolved internally by PySTAC.
stac_io = DaFabStacIO(ca_bundle=CA_BUNDLE, verify_tls=VERIFY_TLS)
pystac.StacIO.set_default(lambda: DaFabStacIO(ca_bundle=CA_BUNDLE, verify_tls=VERIFY_TLS))
print("TLS verification:", "on" if VERIFY_TLS else "off")
print("CA bundle:", CA_BUNDLE or "Python default")

TLS verification: off
CA bundle: Python default


### 1.4 Table rendering

Render long HREFs without pandas truncation so the notebook remains readable and every link stays inspectable.


In [37]:
def rows_as_table(rows: list[dict]) -> None:
    if not rows:
        print("[]")
        return
    # Fall back to plain rows when rich notebook display is unavailable.
    if display is None or HTML is None:
        for row in rows:
            print(row)
        return

    # Escape values while allowing long HREFs to wrap without truncation.
    columns = list(rows[0])
    head = "".join(f"<th>{escape(column)}</th>" for column in columns)
    body = "".join(
        "<tr>"
        + "".join(f"<td>{escape(str(row.get(column, '')))}</td>" for column in columns)
        + "</tr>"
        for row in rows
    )
    display(HTML(f"""
    <style>
      .dafab-table {{ border-collapse: collapse; font-size: 13px; }}
      .dafab-table th, .dafab-table td {{ border: 1px solid #ddd; padding: 4px 6px; vertical-align: top; }}
      .dafab-table td {{ white-space: normal; overflow-wrap: anywhere; max-width: 900px; }}
      .dafab-table th {{ text-align: left; background: #f7f7f7; }}
    </style>
    <table class="dafab-table"><thead><tr>{head}</tr></thead><tbody>{body}</tbody></table>
    """))

### 1.5 STAC helpers

Keep small helpers for raw STAC dictionaries, link filtering, self HREF lookup, and collection IDs encoded in DaFab item URLs.


In [38]:
# Extract a readable identifier without depending on the service hostname.
def last_segment(href: str) -> str:
    return urlparse(href).path.rstrip("/").split("/")[-1]


# Inspect raw links when a relation has no dedicated PySTAC helper.
def self_href(data: dict) -> str | None:
    for link in data.get("links", []):
        if isinstance(link, dict) and link.get("rel") == "self":
            return link.get("href")
    return None


def link_rows(data: dict, rel: str | None = None) -> list[dict]:
    result = data.get("links", [])
    if rel is not None:
        result = [link for link in result if link.get("rel") == rel]
    return result


# Preserve DaFab's published absolute links instead of rewriting them locally.
def raw_dict(obj: pystac.STACObject) -> dict:
    return obj.to_dict(transform_hrefs=False)


# Prefer the canonical self HREF, with the object ID as a final fallback.
def object_href(obj: pystac.STACObject) -> str:
    return obj.get_self_href() or self_href(raw_dict(obj)) or obj.id


# Group discovered Items using the collection segment of the DaFab URL.
def collection_id_from_item_href(href: str) -> str | None:
    parts = [part for part in urlparse(href).path.split("/") if part]
    if "collections" not in parts or "items" not in parts:
        return None
    index = parts.index("collections")
    return parts[index + 1] if index + 1 < len(parts) else None

### 1.6 Core schema validator

Create a local validator for compact per-object schema checks before the fuller PySTAC validation step later in the notebook.


In [39]:
# Select PySTAC's bundled core schema after detecting each object type.
local_validator = LocalValidator()
core_validators = {
    pystac.STACObjectType.CATALOG: local_validator.catalog_validator,
    pystac.STACObjectType.COLLECTION: local_validator.collection_validator,
    pystac.STACObjectType.ITEM: local_validator.item_validator,
}


# Return every message so callers can report a compact count without failing early.
def core_schema_errors(data: dict) -> list[str]:
    object_type = identify_stac_object_type(data)
    if object_type is None:
        return ["not identified as a STAC object"]
    return [error.message for error in core_validators[object_type]().iter_errors(data)]

## 2. Root-first navigation

### 2.1 Open the live DaFab root

Start from the public root catalog. This is the preferred navigation entry point because it exposes the advertised collections and the rest of the tree.


In [40]:
# Read only the root explicitly; all later locations come from published links.
root = pystac.read_file(DAFAB_STAC_URL, stac_io=stac_io)
# Keep an unchanged dictionary view for inspecting the exact root link declarations.
root_data = raw_dict(root)
print(root)

<Catalog id=stac>


### 2.2 Root links

Inspect the root link relations exactly as published by DaFab. These links define the first navigation choices available to PySTAC.


In [41]:
# Display every published relation before asking PySTAC to follow any of them.
rows_as_table(
    [
        {"rel": link.get("rel"), "href": link.get("href"), "target_id": last_segment(link.get("href", ""))}
        for link in root_data.get("links", [])
    ]
)

rel,href,target_id
self,https://dafab.cern.ch/stac,stac
root,https://dafab.cern.ch/stac,stac
child,https://dafab.cern.ch/stac/collections/sentinel_2_l2a,sentinel_2_l2a
child,https://dafab.cern.ch/stac/collections/water_analysis,water_analysis
child,https://dafab.cern.ch/stac/collections/smart_agriculture,smart_agriculture


### 2.3 Discover collections from the root

Use PySTAC's child traversal to resolve the root collection links instead of hardcoding collection URLs.


In [42]:
# get_children resolves the root's child links through the registered StacIO.
collections = list(root.get_children())
# Retain direct lookup while displaying the class and canonical HREF of each result.
collection_by_id = {collection.id: collection for collection in collections}
rows_as_table(
    [
        {"id": collection.id, "class": type(collection).__name__, "self_href": object_href(collection)}
        for collection in collections
    ]
)

id,class,self_href
sentinel_2_l2a,Collection,https://dafab.cern.ch/stac/collections/sentinel_2_l2a
water_analysis,Collection,https://dafab.cern.ch/stac/collections/water_analysis
smart_agriculture,Collection,https://dafab.cern.ch/stac/collections/smart_agriculture


### 2.4 Collection summaries

Summarize each discovered collection and count the outgoing links that drive further navigation.


In [43]:
# Compare direct Item membership with navigation through nested facet catalogs.
collection_rows = []
for collection in collections:
    data = raw_dict(collection)
    collection_rows.append(
        {
            "id": collection.id,
            "license": data.get("license"),
            "spatial_bbox": data.get("extent", {}).get("spatial", {}).get("bbox"),
            "temporal_interval": data.get("extent", {}).get("temporal", {}).get("interval"),
            "child_links": len(link_rows(data, "child")),
            "item_links": len(link_rows(data, "item")),
        }
    )
rows_as_table(collection_rows)

id,license,spatial_bbox,temporal_interval,child_links,item_links
sentinel_2_l2a,proprietary,"[[-120.75467671035365, -38.035683, 147.108611, 54.14801139115367]]","[['2015-07-16T09:10:06.027000Z', '2026-06-06T09:20:29.024000Z']]",0,8181
water_analysis,proprietary,"[[-0.699362547294467, 25.19645858464566, 90.0991685470694, 39.72681533153163]]","[['2015-07-16T09:10:06.027000Z', '2026-06-06T09:20:29.024000Z']]",2,0
smart_agriculture,proprietary,"[[-0.5726919766587134, 25.19645858464566, 90.0991685470694, 54.14800242200601]]","[['2015-07-16T09:10:06.027000Z', '2024-09-21T10:58:50.327000Z']]",1,0


### 2.5 Walk the catalog tree

Walk every reachable collection/catalog node from the root. This reveals the DaFab facet hierarchy and where item links are attached.


In [44]:
# Keep each reached Catalog or Collection once for later validation and lookup.
discovered_nodes_by_href: dict[str, pystac.STACObject] = {}
tree_rows = []

# Traverse the catalog hierarchy but do not consume the Item iterator here.
for node, children_iter, _items_iter in root.walk():
    children = list(children_iter)
    data = raw_dict(node)
    href = object_href(node)
    # The self HREF is a stable key even when IDs repeat in different branches.
    discovered_nodes_by_href[href] = node
    tree_rows.append(
        {
            "id": node.id,
            "class": type(node).__name__,
            "self_href": href,
            "child_ids": [child.id for child in children],
            "item_link_count": len(link_rows(data, "item")),
        }
    )

print("Catalog/Collection nodes discovered:", len(discovered_nodes_by_href))
rows_as_table(tree_rows)

Catalog/Collection nodes discovered: 53


id,class,self_href,child_ids,item_link_count
stac,Catalog,https://dafab.cern.ch/stac,"['sentinel_2_l2a', 'water_analysis', 'smart_agriculture']",0
sentinel_2_l2a,Collection,https://dafab.cern.ch/stac/collections/sentinel_2_l2a,[],8181
water_analysis,Collection,https://dafab.cern.ch/stac/collections/water_analysis,"['water_anomaly', 'water_basin']",0
water_anomaly,Catalog,https://dafab.cern.ch/stac/collections/water_analysis/catalogs/water_anomaly,"['water_anomaly_flood', 'water_anomaly_drought', 'water_anomaly_normal']",0
water_anomaly_flood,Catalog,https://dafab.cern.ch/stac/collections/water_analysis/catalogs/water_anomaly/water_anomaly_flood,[],1460
water_anomaly_drought,Catalog,https://dafab.cern.ch/stac/collections/water_analysis/catalogs/water_anomaly/water_anomaly_drought,[],106
water_anomaly_normal,Catalog,https://dafab.cern.ch/stac/collections/water_analysis/catalogs/water_anomaly/water_anomaly_normal,[],6245
water_basin,Catalog,https://dafab.cern.ch/stac/collections/water_analysis/catalogs/water_basin,"['water_basin_ganges', 'water_basin_hybas_id_l6_2060010050', 'water_basin_hybas_id_l3_2030016230', 'water_basin_ebro', 'water_basin_rhone', 'water_basin_seine', 'water_basin_garonne', 'water_basin_elbe', 'water_basin_po', 'water_basin_dordogne', 'water_basin_buzi', 'water_basin_mekong', 'water_basin_iso_vnm', 'water_basin_chao_phraya', 'water_basin_lam', 'water_basin_red', 'water_basin_ma', 'water_basin_thu_bon', 'water_basin_rhine', 'water_basin_parana', 'water_basin_niger', 'water_basin_oder', 'water_basin_brahmaputra', 'water_basin_loire', 'water_basin_great_salt_lake', 'water_basin_murray', 'water_basin_colorado_mrbid_4405', 'water_basin_san_joaquin', 'water_basin_sittang', 'water_basin_chad', 'water_basin_iso_zaf', 'water_basin_olifants', 'water_basin_iso_aus', 'water_basin_weser', 'water_basin_irrawaddy', 'water_basin_jubba', 'water_basin_brazos', 'water_basin_titicaca', 'water_basin_sabine']",0
water_basin_ganges,Catalog,https://dafab.cern.ch/stac/collections/water_analysis/catalogs/water_basin/water_basin_ganges,[],1
water_basin_hybas_id_l6_2060010050,Catalog,https://dafab.cern.ch/stac/collections/water_analysis/catalogs/water_basin/water_basin_hybas_id_l6_2060010050,[],17


## 3. Item navigation

### 3.1 Item links discovered from the tree

Collect every `item` link advertised by the root-driven tree. The stored rows remain complete; only this visual table is shortened.


In [45]:
# Preserve the advertising parent because one Item may appear under several facets.
item_link_rows = []
for parent_href, node in discovered_nodes_by_href.items():
    for link in link_rows(raw_dict(node), "item"):
        href = link.get("href")
        if href:
            item_link_rows.append(
                {
                    "parent_id": node.id,
                    "parent_class": type(node).__name__,
                    "item_id": last_segment(href),
                    "item_href": href,
                }
            )

# Keep all memberships, but deduplicate HREFs before any Item documents are fetched.
item_link_rows = sorted(item_link_rows, key=lambda row: (row["parent_id"], row["item_id"]))
item_hrefs = sorted({row["item_href"] for row in item_link_rows})
print("Item links discovered:", len(item_link_rows))
print("Unique item HREFs:", len(item_hrefs))

# Shorten only the rendered table; the complete rows remain available for analysis.
shown_item_link_rows = item_link_rows
if len(item_link_rows) > 16:
    shown_item_link_rows = [
        *item_link_rows[:8],
        {
            "parent_id": "...",
            "parent_class": "...",
            "item_id": f"... {len(item_link_rows) - 16} rows omitted ...",
            "item_href": "...",
        },
        *item_link_rows[-8:],
    ]
rows_as_table(shown_item_link_rows)

Item links discovered: 24342
Unique item HREFs: 16531


parent_id,parent_class,item_id,item_href
agriculture_season_autumn,Catalog,S2A_31TDK_20240916_0_L2A_smart_agriculture_120,https://dafab.cern.ch/stac/collections/smart_agriculture/items/S2A_31TDK_20240916_0_L2A_smart_agriculture_120
agriculture_season_autumn,Catalog,S2A_MSIL2A_20151022T034152_N0500_R018_T48PWB_20231009T082656_smart_agriculture_200,https://dafab.cern.ch/stac/collections/smart_agriculture/items/S2A_MSIL2A_20151022T034152_N0500_R018_T48PWB_20231009T082656_smart_agriculture_200
agriculture_season_autumn,Catalog,S2A_MSIL2A_20151022T034152_N0500_R018_T48PWC_20231009T082656_smart_agriculture_200,https://dafab.cern.ch/stac/collections/smart_agriculture/items/S2A_MSIL2A_20151022T034152_N0500_R018_T48PWC_20231009T082656_smart_agriculture_200
agriculture_season_autumn,Catalog,S2A_MSIL2A_20151022T034152_N0500_R018_T48QVD_20231009T082656_smart_agriculture_200,https://dafab.cern.ch/stac/collections/smart_agriculture/items/S2A_MSIL2A_20151022T034152_N0500_R018_T48QVD_20231009T082656_smart_agriculture_200
agriculture_season_autumn,Catalog,S2A_MSIL2A_20151022T034152_N0500_R018_T48QVE_20231009T082656_smart_agriculture_200,https://dafab.cern.ch/stac/collections/smart_agriculture/items/S2A_MSIL2A_20151022T034152_N0500_R018_T48QVE_20231009T082656_smart_agriculture_200
agriculture_season_autumn,Catalog,S2A_MSIL2A_20151022T034152_N0500_R018_T48QWD_20231009T082656_smart_agriculture_200,https://dafab.cern.ch/stac/collections/smart_agriculture/items/S2A_MSIL2A_20151022T034152_N0500_R018_T48QWD_20231009T082656_smart_agriculture_200
agriculture_season_autumn,Catalog,S2A_MSIL2A_20151115T032112_N0500_R075_T48PYA_20231009T042401_smart_agriculture_200,https://dafab.cern.ch/stac/collections/smart_agriculture/items/S2A_MSIL2A_20151115T032112_N0500_R075_T48PYA_20231009T042401_smart_agriculture_200
agriculture_season_autumn,Catalog,S2A_MSIL2A_20151115T032112_N0500_R075_T48QYD_20231009T042401_smart_agriculture_200,https://dafab.cern.ch/stac/collections/smart_agriculture/items/S2A_MSIL2A_20151115T032112_N0500_R075_T48QYD_20231009T042401_smart_agriculture_200
...,...,... 24326 rows omitted ...,...
water_basin_thu_bon,Catalog,S2B_MSIL2A_20230211T030849_N0510_R075_T48PYC_20240813T185708_water_analysis_200,https://dafab.cern.ch/stac/collections/water_analysis/items/S2B_MSIL2A_20230211T030849_N0510_R075_T48PYC_20240813T185708_water_analysis_200


### 3.2 Representative Item HREFs by collection

Pick one discovered Item per collection so the rest of the notebook stays compact while still covering original and derived products.


In [46]:
# Partition the unique Item HREFs without fetching their documents.
item_hrefs_by_collection: dict[str, list[str]] = {}
for href in item_hrefs:
    collection_id = collection_id_from_item_href(href) or "unknown"
    item_hrefs_by_collection.setdefault(collection_id, []).append(href)

# Choose the first sorted HREF for deterministic, low-volume live retrieval.
representative_item_hrefs = [hrefs[0] for _, hrefs in sorted(item_hrefs_by_collection.items())]
rows_as_table(
    [
        {
            "collection": collection_id,
            "discovered_item_links": len(hrefs),
            "representative_item_href": hrefs[0],
        }
        for collection_id, hrefs in sorted(item_hrefs_by_collection.items())
    ]
)

collection,discovered_item_links,representative_item_href
sentinel_2_l2a,8181,https://dafab.cern.ch/stac/collections/sentinel_2_l2a/items/S2A_30TYN_20200311_1_L2A
smart_agriculture,538,https://dafab.cern.ch/stac/collections/smart_agriculture/items/S2A_30TYN_20200311_1_L2A_smart_agriculture_120
water_analysis,7812,https://dafab.cern.ch/stac/collections/water_analysis/items/S2A_45RYJ_20221205_0_L2A_water_analysis_100


### 3.3 Fetch representative Items

Load the representative Items with PySTAC. The source HREFs come from the discovered tree, so this remains a live navigation example.


In [47]:
# Retain both PySTAC objects and untransformed JSON for different inspections below.
representative_items_by_href: dict[str, pystac.Item] = {}
representative_item_json_by_href: dict[str, dict] = {}
item_fetch_rows = []

# Fetch only the deterministic representative selected for each collection.
for href in representative_item_hrefs:
    item = pystac.read_file(href, stac_io=stac_io)
    data = raw_dict(item)
    representative_items_by_href[href] = item
    representative_item_json_by_href[href] = data
    item_fetch_rows.append(
        {
            "id": item.id,
            "collection": item.collection_id,
            "self_href": object_href(item),
            "assets": len(item.assets),
            "links": len(data.get("links", [])),
        }
    )

rows_as_table(item_fetch_rows)

id,collection,self_href,assets,links
S2A_30TYN_20200311_1_L2A,sentinel_2_l2a,https://dafab.cern.ch/stac/collections/sentinel_2_l2a/items/S2A_30TYN_20200311_1_L2A,36,4
S2A_30TYN_20200311_1_L2A_smart_agriculture_120,smart_agriculture,https://dafab.cern.ch/stac/collections/smart_agriculture/items/S2A_30TYN_20200311_1_L2A_smart_agriculture_120,1,4
S2A_45RYJ_20221205_0_L2A_water_analysis_100,water_analysis,https://dafab.cern.ch/stac/collections/water_analysis/items/S2A_45RYJ_20221205_0_L2A_water_analysis_100,10,4


### 3.4 Item-first outgoing links

Starting from each fetched Item, inspect every declared outgoing link. These are the direct navigation options available without scanning the tree again.


In [48]:
# Inspect outgoing declarations without fetching target HREFs in this cell.
item_first_link_rows = []
for source_href, data in representative_item_json_by_href.items():
    for link in data.get("links", []):
        href = link.get("href")
        item_first_link_rows.append(
            {
                "item_id": data.get("id"),
                "collection": data.get("collection"),
                "rel": link.get("rel"),
                "target_id": last_segment(href) if href else None,
                "href": href,
            }
        )

rows_as_table(item_first_link_rows)

item_id,collection,rel,target_id,href
S2A_30TYN_20200311_1_L2A,sentinel_2_l2a,self,S2A_30TYN_20200311_1_L2A,https://dafab.cern.ch/stac/collections/sentinel_2_l2a/items/S2A_30TYN_20200311_1_L2A
S2A_30TYN_20200311_1_L2A,sentinel_2_l2a,root,stac,https://dafab.cern.ch/stac
S2A_30TYN_20200311_1_L2A,sentinel_2_l2a,collection,sentinel_2_l2a,https://dafab.cern.ch/stac/collections/sentinel_2_l2a
S2A_30TYN_20200311_1_L2A,sentinel_2_l2a,related,S2A_30TYN_20200311_1_L2A_smart_agriculture_120,https://dafab.cern.ch/stac/collections/smart_agriculture/items/S2A_30TYN_20200311_1_L2A_smart_agriculture_120
S2A_30TYN_20200311_1_L2A_smart_agriculture_120,smart_agriculture,self,S2A_30TYN_20200311_1_L2A_smart_agriculture_120,https://dafab.cern.ch/stac/collections/smart_agriculture/items/S2A_30TYN_20200311_1_L2A_smart_agriculture_120
S2A_30TYN_20200311_1_L2A_smart_agriculture_120,smart_agriculture,root,stac,https://dafab.cern.ch/stac
S2A_30TYN_20200311_1_L2A_smart_agriculture_120,smart_agriculture,collection,smart_agriculture,https://dafab.cern.ch/stac/collections/smart_agriculture
S2A_30TYN_20200311_1_L2A_smart_agriculture_120,smart_agriculture,derived_from,S2A_30TYN_20200311_1_L2A,https://dafab.cern.ch/stac/collections/sentinel_2_l2a/items/S2A_30TYN_20200311_1_L2A
S2A_45RYJ_20221205_0_L2A_water_analysis_100,water_analysis,self,S2A_45RYJ_20221205_0_L2A_water_analysis_100,https://dafab.cern.ch/stac/collections/water_analysis/items/S2A_45RYJ_20221205_0_L2A_water_analysis_100
S2A_45RYJ_20221205_0_L2A_water_analysis_100,water_analysis,root,stac,https://dafab.cern.ch/stac


### 3.5 Resolve DaFab-local targets from Items

Resolve Item links that point back into DaFab. External links are reported as external instead of fetched, keeping this cell focused on the DaFab STAC graph.


In [49]:
# Cache resolved types and IDs so repeated targets require only one HTTP read.
target_cache: dict[str, tuple[str, str | None]] = {}


def read_dafab_target(href: str) -> tuple[str, str | None]:
    # Report external provenance without leaving the DaFab graph in this example.
    if not href.startswith(DAFAB_STAC_URL):
        return "external", last_segment(href)
    if href not in target_cache:
        target = pystac.read_file(href, stac_io=stac_io)
        target_cache[href] = (type(target).__name__, target.id)
    return target_cache[href]


# Resolve navigation and provenance relations while skipping loaded self targets.
resolved_item_link_rows = []
for source_href, data in representative_item_json_by_href.items():
    for rel in ["root", "collection", "derived_from", "related"]:
        for link in link_rows(data, rel):
            href = link.get("href")
            if not href:
                continue
            target_type, target_id = read_dafab_target(href)
            resolved_item_link_rows.append(
                {
                    "item_id": data.get("id"),
                    "rel": rel,
                    "target_type": target_type,
                    "target_id": target_id,
                    "href": href,
                }
            )

rows_as_table(resolved_item_link_rows)

item_id,rel,target_type,target_id,href
S2A_30TYN_20200311_1_L2A,root,Catalog,stac,https://dafab.cern.ch/stac
S2A_30TYN_20200311_1_L2A,collection,Collection,sentinel_2_l2a,https://dafab.cern.ch/stac/collections/sentinel_2_l2a
S2A_30TYN_20200311_1_L2A,related,Item,S2A_30TYN_20200311_1_L2A_smart_agriculture_120,https://dafab.cern.ch/stac/collections/smart_agriculture/items/S2A_30TYN_20200311_1_L2A_smart_agriculture_120
S2A_30TYN_20200311_1_L2A_smart_agriculture_120,root,Catalog,stac,https://dafab.cern.ch/stac
S2A_30TYN_20200311_1_L2A_smart_agriculture_120,collection,Collection,smart_agriculture,https://dafab.cern.ch/stac/collections/smart_agriculture
S2A_30TYN_20200311_1_L2A_smart_agriculture_120,derived_from,Item,S2A_30TYN_20200311_1_L2A,https://dafab.cern.ch/stac/collections/sentinel_2_l2a/items/S2A_30TYN_20200311_1_L2A
S2A_45RYJ_20221205_0_L2A_water_analysis_100,root,Catalog,stac,https://dafab.cern.ch/stac
S2A_45RYJ_20221205_0_L2A_water_analysis_100,collection,Collection,water_analysis,https://dafab.cern.ch/stac/collections/water_analysis
S2A_45RYJ_20221205_0_L2A_water_analysis_100,derived_from,Item,S2A_45RYJ_20221205_0_L2A,https://dafab.cern.ch/stac/collections/sentinel_2_l2a/items/S2A_45RYJ_20221205_0_L2A


### 3.6 Reverse catalog membership

Item documents do not need to contain backlinks to every catalog that lists them. When that reverse view is needed, derive it from the root-driven tree scan.


In [50]:
# Reverse the earlier Catalog-to-Item links rather than looking for Item backlinks.
representative_item_ids = {data.get("id") for data in representative_item_json_by_href.values()}
# These rows describe facet placement, which is separate from Item provenance.
reverse_membership_rows = [
    {
        "item_id": row["item_id"],
        "linked_by": row["parent_id"],
        "parent_class": row["parent_class"],
        "item_href": row["item_href"],
    }
    for row in item_link_rows
    if row["item_id"] in representative_item_ids
]

rows_as_table(reverse_membership_rows)

item_id,linked_by,parent_class,item_href
S2A_30TYN_20200311_1_L2A_smart_agriculture_120,agriculture_season_spring,Catalog,https://dafab.cern.ch/stac/collections/smart_agriculture/items/S2A_30TYN_20200311_1_L2A_smart_agriculture_120
S2A_30TYN_20200311_1_L2A,sentinel_2_l2a,Collection,https://dafab.cern.ch/stac/collections/sentinel_2_l2a/items/S2A_30TYN_20200311_1_L2A
S2A_45RYJ_20221205_0_L2A_water_analysis_100,water_basin_ganges,Catalog,https://dafab.cern.ch/stac/collections/water_analysis/items/S2A_45RYJ_20221205_0_L2A_water_analysis_100


### 3.7 Navigation capability matrix

| Starting point | Works now | Route | Result |
|---|---|---|---|
| Root catalog | yes | `pystac.read_file(root)` -> child links -> `get_children()` / `walk()` | Collections and facet catalogs |
| Collection or Catalog | yes | `child` and `item` links | Deeper facet catalogs and advertised Items |
| Fetched Item | yes | Declared Item links | Root, collection, provenance, and related products when links exist |
| Fetched Item needing facet placement | yes, through the tree | Scan root tree `item` links and match Item IDs/HREFs | Catalogs that advertise the Item |
| Spatial/time/property query | outside PySTAC core | STAC API search with `pystac-client` if exposed, or DaFab enhanced filters | Query-selected Items |


### 3.8 Item summaries

Compare the fetched Items, including declared provenance and related-product links.


In [51]:
# Compare core spatiotemporal fields, assets, and both provenance directions.
item_rows = []
for href, item in representative_items_by_href.items():
    data = representative_item_json_by_href[href]
    properties = data.get("properties", {})
    item_rows.append(
        {
            "id": item.id,
            "collection": item.collection_id,
            "datetime": properties.get("datetime"),
            "bbox": data.get("bbox"),
            "assets": list(data.get("assets", {}).keys()),
            "related": [last_segment(link["href"]) for link in link_rows(data, "related")],
            "derived_from": [last_segment(link["href"]) for link in link_rows(data, "derived_from")],
        }
    )
rows_as_table(item_rows)

id,collection,datetime,bbox,assets,related,derived_from
S2A_30TYN_20200311_1_L2A,sentinel_2_l2a,2020-03-11T10:59:12.936000Z,"[-0.5726919766587134, 42.30246099916119, 0.8181108416652176, 43.32624770923024]","['aot', 'nir', 'red', 'scl', 'wvp', 'blue', 'green', 'nir08', 'nir09', 'swir16', 'swir22', 'visual', 'aot-jp2', 'coastal', 'nir-jp2', 'red-jp2', 'scl-jp2', 'wvp-jp2', 'blue-jp2', 'rededge1', 'rededge2', 'rededge3', 'green-jp2', 'nir08-jp2', 'nir09-jp2', 'thumbnail', 'swir16-jp2', 'swir22-jp2', 'visual-jp2', 'coastal-jp2', 'rededge1-jp2', 'rededge2-jp2', 'rededge3-jp2', 'granule_metadata', 'tileinfo_metadata', 'dafab-field-boundaries']",['S2A_30TYN_20200311_1_L2A_smart_agriculture_120'],[]
S2A_30TYN_20200311_1_L2A_smart_agriculture_120,smart_agriculture,2020-03-11T10:59:12.936000Z,"[-0.5726919766587134, 42.30246099916119, 0.8181108416652176, 43.32624770923024]",['dafab-field-boundaries'],[],['S2A_30TYN_20200311_1_L2A']
S2A_45RYJ_20221205_0_L2A_water_analysis_100,water_analysis,2022-12-05T00:00:00Z,"[88.98479843212748, 25.19645858464566, 90.0991685470694, 26.20599133191409]","['dafab-water-excess', 'dafab-AI-cloud-mask', 'dafab-water-deficit', 'dafab-ESA-worldcover', 'dafab-AI-observed-water', 'dafab-no-AI-observed-water', 'dafab-AI-difference-water-mask', 'dafab-GFM-reference-water-mask', 'dafab-AI-postprocessed-observed-water', 'dafab-no-AI-postprocessed-observed-water']",[],['S2A_45RYJ_20221205_0_L2A']


### 3.9 Asset summaries

List the assets PySTAC sees on each representative Item. PySTAC inspects these HREFs and metadata; it does not download the asset bytes here.


In [52]:
# Inspect declared asset metadata only; no asset bytes are downloaded.
asset_rows = []
for item in representative_items_by_href.values():
    for key, asset in item.assets.items():
        asset_rows.append(
            {
                "item_id": item.id,
                "asset_key": key,
                "media_type": asset.media_type,
                "title": asset.title,
                "href": asset.href,
            }
        )
rows_as_table(asset_rows)

item_id,asset_key,media_type,title,href
S2A_30TYN_20200311_1_L2A,aot,image/tiff; application=geotiff; profile=cloud-optimized,Aerosol optical thickness (AOT),https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/30/T/YN/2020/3/S2A_30TYN_20200311_1_L2A/AOT.tif
S2A_30TYN_20200311_1_L2A,nir,image/tiff; application=geotiff; profile=cloud-optimized,NIR 1 (band 8) - 10m,https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/30/T/YN/2020/3/S2A_30TYN_20200311_1_L2A/B08.tif
S2A_30TYN_20200311_1_L2A,red,image/tiff; application=geotiff; profile=cloud-optimized,Red (band 4) - 10m,https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/30/T/YN/2020/3/S2A_30TYN_20200311_1_L2A/B04.tif
S2A_30TYN_20200311_1_L2A,scl,image/tiff; application=geotiff; profile=cloud-optimized,Scene classification map (SCL),https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/30/T/YN/2020/3/S2A_30TYN_20200311_1_L2A/SCL.tif
S2A_30TYN_20200311_1_L2A,wvp,image/tiff; application=geotiff; profile=cloud-optimized,Water vapour (WVP),https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/30/T/YN/2020/3/S2A_30TYN_20200311_1_L2A/WVP.tif
S2A_30TYN_20200311_1_L2A,blue,image/tiff; application=geotiff; profile=cloud-optimized,Blue (band 2) - 10m,https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/30/T/YN/2020/3/S2A_30TYN_20200311_1_L2A/B02.tif
S2A_30TYN_20200311_1_L2A,green,image/tiff; application=geotiff; profile=cloud-optimized,Green (band 3) - 10m,https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/30/T/YN/2020/3/S2A_30TYN_20200311_1_L2A/B03.tif
S2A_30TYN_20200311_1_L2A,nir08,image/tiff; application=geotiff; profile=cloud-optimized,NIR 2 (band 8A) - 20m,https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/30/T/YN/2020/3/S2A_30TYN_20200311_1_L2A/B8A.tif
S2A_30TYN_20200311_1_L2A,nir09,image/tiff; application=geotiff; profile=cloud-optimized,NIR 3 (band 9) - 60m,https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/30/T/YN/2020/3/S2A_30TYN_20200311_1_L2A/B09.tif
S2A_30TYN_20200311_1_L2A,swir16,image/tiff; application=geotiff; profile=cloud-optimized,SWIR 1 (band 11) - 20m,https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/30/T/YN/2020/3/S2A_30TYN_20200311_1_L2A/B11.tif


### 3.10 DaFab property blocks

Show the DaFab-specific metadata blocks preserved by PySTAC as extension/custom fields.


In [53]:
# PySTAC preserves namespaced DaFab fields even without typed extension helpers.
domain_rows = []
for href, item in representative_items_by_href.items():
    properties = representative_item_json_by_href[href].get("properties", {})
    for key, value in properties.items():
        if key.startswith("dafab:"):
            domain_rows.append(
                {
                    "item_id": item.id,
                    "domain_block": key,
                    "fields": list(value.keys()) if isinstance(value, dict) else type(value).__name__,
                }
            )
rows_as_table(domain_rows)

item_id,domain_block,fields
S2A_30TYN_20200311_1_L2A,dafab:field_boundaries,"['field_percentage', 'number_of_fields', 'total_field_area(ha)', 'max_area_of_fields(ha)', 'min_area_of_fields(ha)', 'mean_area_of_fields(ha)', 'field_percentage_tile_grid']"
S2A_30TYN_20200311_1_L2A_smart_agriculture_120,dafab:smart-agriculture,"['field_percentage', 'number_of_fields', 'total_field_area(ha)', 'max_area_of_fields(ha)', 'min_area_of_fields(ha)', 'mean_area_of_fields(ha)', 'smart_agriculture_algorithm_version']"
S2A_45RYJ_20221205_0_L2A_water_analysis_100,dafab:water-analysis,"['deficit_water', 'observed_water', 'excess_total_water', 'difference_water_mask', 'deficit_water_observed', 'observed_water_visible', 'reference_seasonal_water', 'reference_permanent_water', 'excess_total_water_observed', 'excess_seasonal_water_observed', 'deficit_water_observed_area(km2)', 'reference_seasonal_water_visible', 'water_analysis_algorithm_version', 'reference_permanent_water_visible', 'trustworthiness_observed_water_mask', 'excess_total_water_observed_area(km2)', 'excess_water_observed_by_permanent_seasonal']"


## 4. Validation and serialization

### 4.1 STAC validation

This validates the STAC JSON already fetched in the notebook: the root catalog, every discovered collection/catalog node, and the representative Items. Each row reports two checks:

- `core_schema_errors`: local JSON-schema errors from PySTAC's `LocalValidator`, using the detected STAC object type.
- `pystac_full_validation`: PySTAC's normal `validate_dict(...)`, including declared `stac_extensions` when their schemas are resolvable.

This does not validate DaFab/Rucio publication policy, reachable asset bytes, or search/filter behavior; the notebook handles the DaFab-specific structural checks separately below.


In [54]:
# Reuse the fetched documents so validation does not add another graph traversal.
validation_docs = {href: raw_dict(obj) for href, obj in discovered_nodes_by_href.items()}
validation_docs.update(representative_item_json_by_href)

validation_rows = []
for href, data in validation_docs.items():
    object_type = identify_stac_object_type(data)
    # Full validation checks core STAC plus each declared extension schema.
    try:
        validate_dict(
            data,
            stac_object_type=object_type,
            stac_version=data.get("stac_version"),
            extensions=data.get("stac_extensions", []),
            href=href,
        )
        full_validation = "ok"
    except Exception as exc:
        full_validation = f"{type(exc).__name__}: {exc}"

    # Report bundled core-schema errors separately from full validation.
    validation_rows.append(
        {
            "id": data.get("id"),
            "type": object_type.value if object_type else None,
            "core_schema_errors": len(core_schema_errors(data)),
            "pystac_full_validation": full_validation,
        }
    )
rows_as_table(validation_rows)

id,type,core_schema_errors,pystac_full_validation
stac,Catalog,0,ok
sentinel_2_l2a,Collection,0,ok
water_analysis,Collection,0,ok
water_anomaly,Catalog,0,ok
water_anomaly_flood,Catalog,0,ok
water_anomaly_drought,Catalog,0,ok
water_anomaly_normal,Catalog,0,ok
water_basin,Catalog,0,ok
water_basin_ganges,Catalog,0,ok
water_basin_hybas_id_l6_2060010050,Catalog,0,ok


### 4.2 Lightweight DaFab profile constants

Define the DaFab-specific assumptions checked here: expected collections, which collections are derived products, and the structural rules that are not part of generic STAC schema validation.


In [55]:
# Layer a small set of DaFab expectations on top of generic STAC validity.
EXPECTED_COLLECTIONS = {"sentinel_2_l2a", "water_analysis", "smart_agriculture"}
DERIVED_COLLECTIONS = {"water_analysis", "smart_agriculture"}


# Accumulate structured findings so all failed checks can be shown together.
def add_finding(findings: list[dict], href: str, doc_id: str | None, check: str, detail: str) -> None:
    findings.append({"href": href, "id": doc_id, "check": check, "detail": detail})

### 4.3 Root and collection profile checks

These checks cover DaFab graph shape at the top of the tree: expected root children, root backlinks on collections, and facet-catalog children under derived collections.


In [56]:
# Check the DaFab entry-point shape that core STAC intentionally leaves open.
def root_and_collection_findings() -> list[dict]:
    findings = []
    root_children = {collection.id for collection in collections}

    for collection_id in sorted(EXPECTED_COLLECTIONS - root_children):
        add_finding(findings, object_href(root), root.id, "root child collection", f"missing {collection_id}")

    for collection in collections:
        data = raw_dict(collection)
        href = object_href(collection)
        if collection.id not in EXPECTED_COLLECTIONS:
            add_finding(findings, href, collection.id, "collection id", f"unexpected id {collection.id!r}")
        if not link_rows(data, "root"):
            add_finding(findings, href, collection.id, "root link", "missing root link")
        # Derived collections expose products through facet catalog children.
        if collection.id in DERIVED_COLLECTIONS and not link_rows(data, "child"):
            add_finding(findings, href, collection.id, "facet links", "derived collection has no facet catalog children")

    return findings

### 4.4 Item profile checks

These checks cover representative Item expectations: known collection ID, geometry and bbox, derived-product provenance, derived-product assets, observed facet membership, and related links from source Items.


In [57]:
# Apply sampled DaFab publication expectations to the representative Items.
def item_findings() -> list[dict]:
    findings = []
    # Membership comes from downward Catalog-to-Item links found during traversal.
    catalog_item_targets = {row["item_id"] for row in item_link_rows}

    for href, data in representative_item_json_by_href.items():
        item_id = data.get("id")
        collection = data.get("collection")
        if collection not in EXPECTED_COLLECTIONS:
            add_finding(findings, href, item_id, "item collection", f"unexpected collection {collection!r}")
        if not data.get("geometry") or not data.get("bbox"):
            add_finding(findings, href, item_id, "spatial fields", "missing geometry or bbox")
        # Derived products need source provenance, assets, and observed facet placement.
        if collection in DERIVED_COLLECTIONS and not link_rows(data, "derived_from"):
            add_finding(findings, href, item_id, "provenance", "derived item has no derived_from link")
        if collection in DERIVED_COLLECTIONS and not data.get("assets"):
            add_finding(findings, href, item_id, "assets", "derived item has no assets")
        if collection in DERIVED_COLLECTIONS and item_id not in catalog_item_targets:
            add_finding(findings, href, item_id, "facet membership", "derived item is not linked by any fetched facet catalog")
        # Source Items expose reverse provenance through related-product links.
        if collection == "sentinel_2_l2a" and not link_rows(data, "related"):
            add_finding(findings, href, item_id, "provenance", "source item has no related derived-item links")

    return findings

### 4.5 DaFab profile-check result

Display only findings from the lightweight DaFab profile checks; an empty result means the sampled live graph matched those checks.


In [58]:
# An empty combined list means the sampled live graph passed this lightweight profile.
profile_findings = [*root_and_collection_findings(), *item_findings()]
if profile_findings:
    rows_as_table(profile_findings)
else:
    print("No findings from the lightweight DaFab profile checks in this notebook.")

No findings from the lightweight DaFab profile checks in this notebook.


### 4.6 Round-trip serialization

Confirm that fetched objects can be serialized back to dictionaries. `transform_hrefs=False` preserves DaFab's published absolute links exactly.


In [59]:
# Exercise the same graph objects that were inspected and validated above.
roundtrip_objects = {**discovered_nodes_by_href, **representative_items_by_href}
roundtrip_rows = []
for href, obj in roundtrip_objects.items():
    # Explicit mode must preserve the absolute HREFs published by DaFab.
    obj.to_dict(transform_hrefs=False)
    # Test graph-aware default HREF transformation as a separate behavior.
    try:
        obj.to_dict()
        default_status = "ok"
    except Exception as exc:
        default_status = f"{type(exc).__name__}: {exc}"
    roundtrip_rows.append(
        {
            "id": raw_dict(obj).get("id"),
            "without_href_transform": "ok",
            "default_to_dict": default_status,
        }
    )
rows_as_table(roundtrip_rows)

id,without_href_transform,default_to_dict
stac,ok,ok
sentinel_2_l2a,ok,ok
water_analysis,ok,ok
water_anomaly,ok,ok
water_anomaly_flood,ok,ok
water_anomaly_drought,ok,ok
water_anomaly_normal,ok,ok
water_basin,ok,ok
water_basin_ganges,ok,ok
water_basin_hybas_id_l6_2060010050,ok,ok
